## Configuration

**Student**: Ahmed  
**Student ID**: 202201166

In [ ]:
# ============================================================
# YOUR INFORMATION (Already configured)
# ============================================================
EXPERIMENT_NAME = "Assignment3_Ahmed"
STUDENT_ID = "202201166"
# ============================================================

## Import Libraries

In [ ]:
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

# Set MLflow tracking URI to use SQLite database (same as UI)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# GPU Setup
if torch.cuda.is_available():
    # Enable cuDNN auto-tuner for optimized performance
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.enabled = True
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("No GPU detected, using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"MLflow version: {mlflow.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

## Define the CNN Model

A simple Convolutional Neural Network for CIFAR-10 classification.

In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for CIFAR-10 classification."""
    
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(self.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = self.pool(self.relu(self.conv3(x)))  # 8x8 -> 4x4
        x = x.view(-1, 64 * 4 * 4)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

## Data Loading Functions

In [ ]:
def get_data_loaders(batch_size, use_cuda=False):
    """Create train and test data loaders for CIFAR-10 with GPU optimization."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    train_dataset = datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    test_dataset = datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )

    # GPU optimization: pin_memory speeds up CPU to GPU transfer
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=use_cuda  # Faster data transfer to GPU
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_cuda
    )

    return train_loader, test_loader

## Training and Evaluation Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch and return average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        # non_blocking=True allows async CPU to GPU transfer when using pin_memory
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def evaluate(model, test_loader, criterion, device):
    """Evaluate the model on test data."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            # non_blocking=True allows async CPU to GPU transfer
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

## Main Training Function with MLflow Integration

This function demonstrates all the MLflow pillars:
- **Experiment Name**: Groups related runs together
- **Run Context**: `mlflow.start_run()` to group all logs
- **Parameters**: Hyperparameters logged with `mlflow.log_param()`
- **Tags**: Metadata logged with `mlflow.set_tag()`
- **Metrics**: Live logging with `mlflow.log_metric()` at each epoch
- **Model Artifact**: Saved using `mlflow.pytorch.log_model()`

In [ ]:
def train_with_mlflow(learning_rate, epochs, batch_size, optimizer_name, run_name=None):
    """
    Train a model with full MLflow tracking (GPU optimized).
    
    Args:
        learning_rate: Learning rate for the optimizer
        epochs: Number of training epochs
        batch_size: Batch size for data loading
        optimizer_name: Name of optimizer ('Adam', 'SGD', or 'RMSprop')
        run_name: Optional name for this MLflow run
    """
    # Set device with GPU optimization
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"
    print(f"Using device: {device}")
    
    if use_cuda:
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    # Set the MLflow experiment name
    mlflow.set_experiment(EXPERIMENT_NAME)

    # Start MLflow run - all logs will be grouped together
    with mlflow.start_run(run_name=run_name):
        
        # ========================================
        # LOG TAGS - Identify your work
        # ========================================
        mlflow.set_tag("student_id", STUDENT_ID)
        mlflow.set_tag("model_type", "SimpleCNN")
        mlflow.set_tag("dataset", "CIFAR-10")
        mlflow.set_tag("framework", "PyTorch")

        # ========================================
        # LOG PARAMETERS - At least 3 hyperparameters
        # ========================================
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("optimizer", optimizer_name)
        mlflow.log_param("device", str(device))
        
        # Log GPU info if available
        if use_cuda:
            mlflow.log_param("gpu_name", torch.cuda.get_device_name(0))
            mlflow.log_param("cuda_version", torch.version.cuda)

        print(f"\n{'='*60}")
        print(f"MLflow Run Started")
        print(f"Experiment: {EXPERIMENT_NAME}")
        print(f"Run Name: {run_name}")
        print(f"{'='*60}")
        print(f"Hyperparameters:")
        print(f"  Learning Rate: {learning_rate}")
        print(f"  Epochs: {epochs}")
        print(f"  Batch Size: {batch_size}")
        print(f"  Optimizer: {optimizer_name}")
        print(f"{'='*60}\n")

        # Get data loaders with GPU optimization
        train_loader, test_loader = get_data_loaders(batch_size, use_cuda=use_cuda)

        # Initialize model, criterion, and optimizer
        model = SimpleCNN().to(device)
        criterion = nn.CrossEntropyLoss()

        if optimizer_name == "Adam":
            optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        elif optimizer_name == "SGD":
            optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
        else:
            optimizer = optim.RMSprop(model.parameters(), lr=learning_rate)

        # Training loop with LIVE LOGGING
        best_test_accuracy = 0.0
        start_time = time.time()

        for epoch in range(1, epochs + 1):
            # Train
            train_loss, train_accuracy = train_epoch(
                model, train_loader, criterion, optimizer, device
            )

            # Evaluate
            test_loss, test_accuracy = evaluate(
                model, test_loader, criterion, device
            )

            # ========================================
            # LIVE LOGGING - Log metrics at every epoch
            # This generates the "Learning Curve" graphs
            # ========================================
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("train_accuracy", train_accuracy, step=epoch)
            mlflow.log_metric("test_loss", test_loss, step=epoch)
            mlflow.log_metric("test_accuracy", test_accuracy, step=epoch)

            # Track best accuracy
            if test_accuracy > best_test_accuracy:
                best_test_accuracy = test_accuracy

            print(f"Epoch [{epoch}/{epochs}] "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}% | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.2f}%")

        training_time = time.time() - start_time

        # Log final summary metrics
        mlflow.log_metric("best_test_accuracy", best_test_accuracy)
        mlflow.log_metric("final_train_loss", train_loss)
        mlflow.log_metric("final_test_loss", test_loss)
        mlflow.log_metric("training_time_seconds", training_time)

        # ========================================
        # SAVE MODEL - Using MLflow Model Flavor
        # This saves model weights and environment details
        # ========================================
        print("\nSaving model to MLflow...")
        mlflow.pytorch.log_model(
            model,
            "model",
        )

        print(f"\n{'='*60}")
        print(f"Training Complete!")
        print(f"Best Test Accuracy: {best_test_accuracy:.2f}%")
        print(f"Training Time: {training_time:.2f} seconds")
        print(f"Run ID: {mlflow.active_run().info.run_id}")
        print(f"{'='*60}")
        
        return best_test_accuracy

## Download CIFAR-10 Dataset

Download the dataset before running experiments.

In [ ]:
# Download CIFAR-10 dataset
print("Downloading CIFAR-10 dataset...")
_ = get_data_loaders(64, use_cuda=torch.cuda.is_available())
print("Dataset ready!")

---

## Phase 3: Run 5 Experiments with Different Hyperparameters

We'll vary:
- **Learning rates**: 0.001, 0.01, 0.1
- **Batch sizes**: 64, 128
- **Optimizers**: Adam, SGD

**IMPORTANT**: Make sure MLflow UI is running in a separate terminal before proceeding!
```bash
mlflow ui --port 5000
```

### Run 1: Baseline (LR=0.001, BS=64, Adam)

In [ ]:
train_with_mlflow(
    learning_rate=0.001,
    epochs=10,
    batch_size=64,
    optimizer_name="Adam",
    run_name="Run1_LR_0.001_BS_64_Adam"
)

### Run 2: Higher Learning Rate (LR=0.01)

In [ ]:
train_with_mlflow(
    learning_rate=0.01,
    epochs=10,
    batch_size=64,
    optimizer_name="Adam",
    run_name="Run2_LR_0.01_BS_64_Adam"
)

### Run 3: Very High Learning Rate (LR=0.1) - May Show Instability!

In [ ]:
train_with_mlflow(
    learning_rate=0.1,
    epochs=10,
    batch_size=64,
    optimizer_name="Adam",
    run_name="Run3_LR_0.1_BS_64_Adam"
)

### Run 4: Larger Batch Size (BS=128)

In [ ]:
train_with_mlflow(
    learning_rate=0.001,
    epochs=10,
    batch_size=128,
    optimizer_name="Adam",
    run_name="Run4_LR_0.001_BS_128_Adam"
)

### Run 5: SGD Optimizer with Momentum

In [ ]:
train_with_mlflow(
    learning_rate=0.01,
    epochs=10,
    batch_size=64,
    optimizer_name="SGD",
    run_name="Run5_LR_0.01_BS_64_SGD"
)

---

## Query MLflow Programmatically (Optional)

You can also query your experiment results programmatically:

In [ ]:
import mlflow
import pandas as pd

# Get experiment by name
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment:
    print(f"Experiment ID: {experiment.experiment_id}")
    print(f"Artifact Location: {experiment.artifact_location}")
    
    # Search for all runs in this experiment
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.best_test_accuracy DESC"]
    )
    
    # Display summary
    print(f"\nTotal Runs: {len(runs)}")
    print("\nRuns sorted by Best Test Accuracy:")
    print("="*80)
    
    for idx, row in runs.iterrows():
        print(f"Run: {row.get('tags.mlflow.runName', 'N/A')}")
        print(f"  Best Test Accuracy: {row.get('metrics.best_test_accuracy', 'N/A'):.2f}%")
        print(f"  Learning Rate: {row.get('params.learning_rate', 'N/A')}")
        print(f"  Batch Size: {row.get('params.batch_size', 'N/A')}")
        print(f"  Optimizer: {row.get('params.optimizer', 'N/A')}")
        print()
else:
    print(f"Experiment '{EXPERIMENT_NAME}' not found. Run the training cells first!")